# Day 1 — Frozen graph and exact reference

This notebook is a short teaching narrative. All scientific logic is imported from reusable modules in `src/`.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data/graph.json').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from graph import DEFAULT_GRAPH_PATH, edge_order_records, load_graph
from exact_reference import enumerate_simple_paths_independent, networkx_shortest_reference

graph = load_graph(DEFAULT_GRAPH_PATH)
graph.number_of_nodes(), graph.number_of_edges(), graph.graph['source'], graph.graph['target']

## Frozen edge/qubit ordering

Character `i` in every future edge bitstring refers to the edge with `qubit_index=i`. The order comes from JSON, not NetworkX iteration.

In [ ]:
edge_order_records(graph)

## Two exact methods

Method A uses NetworkX's weighted shortest-path routine. Method B uses the project's deterministic DFS to enumerate every simple directed source-to-target path and independently accumulate weights.

In [ ]:
networkx_reference = networkx_shortest_reference(graph)
all_routes = enumerate_simple_paths_independent(graph)
enumeration_reference = all_routes[0]
assert networkx_reference == enumeration_reference
networkx_reference

In [ ]:
[{'rank': rank, 'path': route.node_path, 'cost': route.cost} for rank, route in enumerate(all_routes, start=1)]

## Figure 1

The exact shortest route is highlighted; all non-optimal directed edges are visually subdued.

In [ ]:
from IPython.display import Image, display
display(Image(filename=PROJECT_ROOT / 'figures/01_frozen_weighted_graph.png'))

In [ ]:
second_best_cost = min(route.cost for route in all_routes if route.cost > enumeration_reference.cost)
{
    'exact_node_path': enumeration_reference.node_path,
    'exact_edge_path': enumeration_reference.edge_path,
    'edge_bitstring': enumeration_reference.bitstring_text,
    'C_star': enumeration_reference.cost,
    'simple_path_count': len(all_routes),
    'second_best_cost': second_best_cost,
    'optimality_gap': second_best_cost - enumeration_reference.cost,
}